# キャッサバの病害分類：ResNet-18の最小サンプル

画像の確認 → データ分割 → 学習 → テスト → 誤分類の分析を一巡するノートブックです。
PyTorch / TorchVision の **ImageNet事前学習済みResNet-18** を5クラス分類に変更し、全層を追加学習します。
自作の外部Pythonファイルには依存しません。上から順に実行してください。

## データと環境の準備

[Kaggle: Cassava Leaf Disease Classification](https://www.kaggle.com/competitions/cassava-leaf-disease-classification/data)
からデータを取得・展開し、次のように配置します。Kaggle側でログイン・規約への同意が必要な場合があります。
データの取得はこのノートブックでは自動実行しません。

```text
B3CNNpart/
├── Cassava_ResNet18_sample.ipynb
└── data/cassava/
    ├── train.csv                       # image_id, label の2列
    ├── label_num_to_disease_map.json   # ラベル番号と病名の対応
    └── train_images/
        ├── 1000015157.jpg
        └── ...
```

PyTorchとTorchVisionは [公式の手順](https://pytorch.org/get-started/locally/) で対応する組み合わせを入れ、
残りは `python -m pip install -r requirements-cassava.txt` で準備します。
初回の事前学習済み重みの取得にはネット接続が必要です。NVIDIA GPUがなければCPUを使います。

初期設定は **各クラス最大200枚・5 epoch** です。流れの確認用であり、性能の上限を調べる設定ではありません。


## 1. ライブラリと設定

まず変更するのは `DATA_DIR`、`MAX_PER_CLASS`、`EPOCHS`、`BATCH_SIZE`、`LR` です。
`MAX_PER_CLASS = None` にすると全画像を使います。結果は実行時刻ごとのフォルダに保存します。


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

PROJECT_DIR = Path.cwd() / "B3CNNpart" if (Path.cwd() / "B3CNNpart").is_dir() else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "cassava"  # 手元の配置に合わせて変更
OUTPUT_DIR = PROJECT_DIR / "outputs" / "cassava_resnet18" / datetime.now().strftime("%Y%m%d_%H%M%S")
SEED = 42
MAX_PER_CLASS = 200  # Noneなら全画像。最初は少量で流れを確認
EPOCHS = 5
BATCH_SIZE = 32
LR = 1e-4

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("data:", DATA_DIR.resolve())
print("output:", OUTPUT_DIR.resolve())


## 2. CSVを読み、クラス数と画像を確認する

病名の対応は付属のJSONから読みます。まず元データのクラス別枚数を確認します。
サンプルでは各クラス最大200枚に絞るため、後の評価値は元データ全体のクラス比率とは異なる場合があります。


In [ ]:
required = [DATA_DIR / "train.csv", DATA_DIR / "label_num_to_disease_map.json", DATA_DIR / "train_images"]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("データを配置するかDATA_DIRを変更してください:\n" + "\n".join(missing))

df = pd.read_csv(DATA_DIR / "train.csv")
assert {"image_id", "label"}.issubset(df.columns), "CSVにはimage_id列とlabel列が必要です"
assert df[["image_id", "label"]].notna().all().all(), "画像名・ラベルに欠損があります"
assert df["image_id"].is_unique, "CSVに重複する画像名があります"
label_map = json.loads((DATA_DIR / "label_num_to_disease_map.json").read_text(encoding="utf-8"))
num_classes = len(label_map)
assert num_classes == 5 and set(map(int, label_map)) == set(range(5)), "0～4の5クラスを想定しています"
assert set(df["label"]) == set(range(num_classes)), "CSVとJSONのラベルが一致しません"
df["label"] = df["label"].astype(int)
class_names = [label_map[str(i)] for i in range(num_classes)]
short_names = [name.split("(")[-1].rstrip(")") if "(" in name else name for name in class_names]

counts = df["label"].value_counts().sort_index()
print(pd.DataFrame({"disease": class_names, "original_count": counts.to_numpy()}))
counts.rename(index=dict(enumerate(short_names))).plot.bar(title="Original class counts", rot=30)
plt.ylabel("Images")
plt.tight_layout()
plt.show()

if MAX_PER_CLASS is not None:
    df = pd.concat([
        group.sample(n=min(MAX_PER_CLASS, len(group)), random_state=SEED)
        for _, group in df.groupby("label")
    ], ignore_index=True)
df = df.reset_index(drop=True)
assert df["label"].value_counts().min() >= 10, "各クラス最低10枚以上を用意してください"
missing_images = [name for name in df["image_id"] if not (DATA_DIR / "train_images" / name).is_file()]
if missing_images:
    raise FileNotFoundError(f"画像が見つかりません（先頭5件）: {missing_images[:5]}")
print("今回使う枚数:", len(df))

fig, axes = plt.subplots(1, num_classes, figsize=(15, 3))
for label, ax in enumerate(axes):
    row = df[df["label"] == label].iloc[0]
    with Image.open(DATA_DIR / "train_images" / row["image_id"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(short_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. 学習70%・検証15%・テスト15%に分ける

クラスの割合を保つ層化分割を使います。学習は重みの更新、検証は最良epochの選択、
テストは最後の評価に使います。Kaggleのラベルなしtest画像ではなく、ラベル付きtrain画像の一部をテスト用に確保します。

ここでは画像単位の分割です。同じ株や連写画像の情報がある場合は、発展時に株・撮影単位で分割してください。


In [ ]:
train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df["label"], random_state=SEED)
train_df, val_df, test_df = [part.reset_index(drop=True) for part in (train_df, val_df, test_df)]
assert set(train_df.image_id).isdisjoint(val_df.image_id)
assert set(train_df.image_id).isdisjoint(test_df.image_id)
assert set(val_df.image_id).isdisjoint(test_df.image_id)

split_counts = pd.DataFrame({
    name: part["label"].value_counts().sort_index()
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]
})
split_counts.index = short_names
print(split_counts)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.concat([part.assign(split=name) for name, part in [
    ("train", train_df), ("val", val_df), ("test", test_df)
]]).to_csv(OUTPUT_DIR / "splits.csv", index=False)


## 4. 画像の前処理とDataLoader

入力はRGBの224×224です。学習時はランダムな切り出し・左右反転、検証とテストは同じ固定の前処理を使います。
ImageNetの平均・標準偏差で正規化します。`num_workers=0` はNotebookで扱いやすい設定です。


In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = weights.transforms()  # 短辺256へリサイズ → 中央224を切り出し → 正規化

class CassavaDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(DATA_DIR / "train_images" / row["image_id"]) as image:
            image = image.convert("RGB")
        return self.transform(image), int(row["label"])

def make_loader(frame, transform, shuffle=False):
    return DataLoader(CassavaDataset(frame, transform), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

val_loader = make_loader(val_df, eval_transform)
test_loader = make_loader(test_df, eval_transform)  # 予測と画像名の対応を保つためshuffleしない


## 5. ResNet-18を学習する

最後の全結合層を5クラス用に交換し、全層を学習します。
損失は交差エントロピー、最適化はAdam（学習率0.0001）、バッチサイズ32です。
毎epoch検証し、**検証損失が最小の重み**を採用します。テストデータは重みの選択には使いません。
このセルを再実行すると、モデルを初期化して最初から学習します。


In [ ]:
def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_sum, correct, count = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)
    return loss_sum / count, correct / count

set_seed(SEED)
train_loader = make_loader(train_df, train_transform, shuffle=True)
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = []
best_loss, best_epoch, best_state = float("inf"), None, None

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    train_loss, train_acc = run_epoch(train_loader, optimizer)
    val_loss, val_acc = run_epoch(val_loader)
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                        train_accuracy=train_acc, val_accuracy=val_acc))
    if val_loss < best_loss:
        best_loss, best_epoch = val_loss, epoch
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    print(f"{epoch:2d}/{EPOCHS}  loss={train_loss:.4f}/{val_loss:.4f}  "
          f"accuracy={train_acc:.3f}/{val_acc:.3f}  {time.perf_counter()-started:.1f}s")

assert best_state is not None, "EPOCHSを1以上にし、学習損失が有限か確認してください"
model.load_state_dict(best_state)
print("採用epoch:", best_epoch)
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)


## 6. 学習曲線を見る

学習損失だけ下がって検証損失が上がる場合は、過学習の可能性があります。
両方の損失が下がらない場合は、画像・ラベル・学習率・epoch数を確認します。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for split in ["train", "val"]:
    axes[0].plot(history_df.epoch, history_df[f"{split}_loss"], label=split)
    axes[1].plot(history_df.epoch, history_df[f"{split}_accuracy"], label=split)
for ax, title in zip(axes, ["Loss", "Accuracy"]):
    ax.set(xlabel="Epoch", title=title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=150)
plt.show()


## 7. テストの正解率・クラス別指標・混同行列

Accuracyは全体の正解率、macro F1は各クラスのF1を均等に平均した値です。
Precisionは「そのクラスと予測した中の正解割合」、Recallは「そのクラスの画像を見つけられた割合」です。
混同行列は縦が正解、横が予測です。対角線以外の大きな値から、混同しやすい病害を探します。


In [ ]:
model.eval()
probabilities = []
with torch.inference_mode():
    for images, _ in test_loader:
        probabilities.append(model(images.to(device)).softmax(dim=1).cpu().numpy())
probabilities = np.concatenate(probabilities)
y_true = test_df["label"].to_numpy()
y_pred = probabilities.argmax(axis=1)
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=list(range(num_classes)), average="macro", zero_division=0)
print(f"Test accuracy: {accuracy:.4f}  /  Macro F1: {macro_f1:.4f}")
report = classification_report(y_true, y_pred, labels=list(range(num_classes)),
                               target_names=short_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame({key: value for key, value in report.items() if isinstance(value, dict)}).T
print(report_df.round(3))
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=list(range(num_classes)),
                                       display_labels=short_names, cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Test confusion matrix")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

results = test_df.copy()
results["prediction"] = y_pred
results["confidence"] = probabilities.max(axis=1)
results["correct"] = y_true == y_pred
for label in range(num_classes):
    results[f"prob_{label}"] = probabilities[:, label]
results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


## 8. 誤分類画像を確認する

予測スコアが高いのに間違えた画像を最大9枚表示します。スコアはsoftmaxの値であり、正しさの保証ではありません。
葉の位置・病斑・背景・暗さ・複数の葉などを観察します。表示する画像は切り出し前の元画像です。


In [ ]:
errors = results[~results["correct"]].sort_values("confidence", ascending=False).head(9)
if errors.empty:
    print("このテスト分割では誤分類がありませんでした。")
else:
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, row) in zip(axes.flat, errors.iterrows()):
        with Image.open(DATA_DIR / "train_images" / row["image_id"]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"True: {short_names[int(row['label'])]}\n"
                     f"Pred: {short_names[int(row['prediction'])]} ({row['confidence']:.2f})")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "misclassified.png", dpi=150)
    plt.show()


## 9. 重みを保存し、1枚を分類する

保存するのは検証損失が最小だった重みです。画像の正規化・ラベル対応も合わせて記録します。
最後の `IMAGE_PATH` を手元の画像に変更すると、その1枚を分類できます。


In [ ]:
torch.save({"model_state_dict": best_state, "class_names": class_names,
            "architecture": "resnet18", "best_epoch": best_epoch,
            "preprocessing": "ResNet18_Weights.IMAGENET1K_V1.transforms()"},
           OUTPUT_DIR / "resnet18_cassava.pth")
summary = {
    "seed": SEED, "max_per_class": MAX_PER_CLASS, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "learning_rate": LR, "input_size": 224,
    "optimizer": "Adam", "pretrained": "IMAGENET1K_V1", "best_epoch": best_epoch,
    "selection_metric": "validation_loss", "class_names": class_names,
    "data_dir": str(DATA_DIR.resolve()), "device": str(device), "torch_version": str(torch.__version__),
    "sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "test_accuracy": float(accuracy), "test_macro_f1": float(macro_f1),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存先:", OUTPUT_DIR.resolve())

def predict_one(image_path):
    with Image.open(image_path) as image:
        image = image.convert("RGB")
    model.eval()
    with torch.inference_mode():
        scores = model(eval_transform(image).unsqueeze(0).to(device)).softmax(dim=1)[0].cpu().numpy()
    return pd.Series(scores, index=class_names, name="score").sort_values(ascending=False)

IMAGE_PATH = DATA_DIR / "train_images" / test_df.iloc[0]["image_id"]
print(predict_one(IMAGE_PATH))


## 次に試すこと

一度に多くを変更せず、変更前後で同じ分割・seedを使って比較します。

1. `MAX_PER_CLASS = None` にして全画像を使い、クラスの偏りとクラス別Recallを確認する。
2. epoch数・学習率・画像拡張を1つずつ変え、検証結果で選ぶ。
3. 複数seedで学習し、平均とばらつきを調べる。

テスト結果を見て繰り返し調整すると、テストが最終評価として使えなくなります。
調整は検証データで行い、必要なら新しい最終評価用データを用意してください。

参考：[TorchVision ResNet-18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)、
[PyTorch 転移学習チュートリアル](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)。
